# Punto 1. Analisis preliminar del problema

Este bloque contiene:
- Carga de librerias y dataset
- 1.a Preprocesamiento del target a dos etiquetas (0 y 1)
- 1.b Clasificacion de variables por tipo

In [14]:
# Fragmento 1: librerias y carga de la base de datos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

# Cargar dataset desde Hugging Face
dataset = load_dataset("YominE/Muscle_Fatigue_Cycling")
train_ds = dataset["train"]

# to_pandas() puede fallar por incompatibilidad pyarrow/pandas en algunos entornos.
# Fallback seguro: construir DataFrame desde las columnas del Dataset.
try:
    df = train_ds.to_pandas()
except ModuleNotFoundError as e:
    if "pandas.api.internals" in str(e):
        df = pd.DataFrame({col: train_ds[col] for col in train_ds.column_names})
    else:
        raise

print(f"Registros: {df.shape[0]} | Variables: {df.shape[1]}")
print("Columnas:", list(df.columns))
df.head()

Registros: 3002137 | Variables: 10
Columnas: ['Time', 'Right Rectus femoris', 'Left Gluteus maximus', 'Left Gastrocnemius medialis', 'Left Semitendinosus', 'Left Biceps femoris caput longus', 'Right Vastus medialis', 'Right Tibialis anterior', 'Left Gastrocnemius lateralis', 'Target']


,Time,Right Rectus femoris,Left Gluteus maximus,Left Gastrocnemius medialis,Left Semitendinosus,Left Biceps femoris caput longus,Right Vastus medialis,Right Tibialis anterior,Left Gastrocnemius lateralis,Target
0,0.000,-0.000264,-0.000015,0.000344,0.000108,0.000182,0.000401,0.000267,-0.000236,0
1,0.001,-0.001002,-0.000045,0.001342,0.000429,0.000712,0.002234,0.001234,-0.001108,0
2,0.002,-0.002173,-0.000034,0.002944,0.001133,0.001692,0.007634,0.003457,-0.003277,0
3,0.003,-0.002676,0.000185,0.003504,0.002319,0.002820,0.017656,0.006587,-0.006940,0
4,0.004,-0.000844,0.000785,0.000426,0.003950,0.003729,0.028542,0.008889,-0.011310,0


In [16]:
# Fragmento 2 (punto 1.a): convertir target a 2 etiquetas (0 y 1)

# Detectar columna objetivo (ajusta manualmente si deseas otro nombre)
possible_target_names = ["target", "label", "labels", "class", "y", "fatigue", "fatigue_state"]
target_col = next((c for c in possible_target_names if c in df.columns), df.columns[-1])

# Limpieza + normalizacion del target
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

# Regla solicitada: etiqueta 2 -> 1
df[target_col] = df[target_col].replace({2: 1})

# Asegurar codificacion binaria final: 0 = normal, 1 = desgaste
df[target_col] = np.where(df[target_col] == 0, 0, 1)

print(f"Columna target usada: {target_col}")
print("Distribucion del target binario:")
display(df[target_col].value_counts().sort_index())

Columna target usada: Target
Distribucion del target binario:


Target
0    2127600
1     874537
Name: count, dtype: int64

In [15]:
# Fragmento 3 (punto 1.b): clasificacion de variables por tipo

feature_cols = [c for c in df.columns if c != target_col]

def infer_variable_type(series: pd.Series) -> str:
    nunique = series.nunique(dropna=True)
    if pd.api.types.is_bool_dtype(series):
        return "binaria"
    if pd.api.types.is_numeric_dtype(series):
        if nunique == 2:
            return "binaria"
        return "numerica"
    if pd.api.types.is_categorical_dtype(series) or pd.api.types.is_object_dtype(series):
        if nunique == 2:
            return "binaria"
        return "categorica"
    return "otro"

summary = pd.DataFrame({
    "variable": feature_cols,
    "dtype": [str(df[c].dtype) for c in feature_cols],
    "n_unicos": [df[c].nunique(dropna=True) for c in feature_cols],
    "faltantes": [df[c].isna().sum() for c in feature_cols],
    "tipo_sugerido": [infer_variable_type(df[c]) for c in feature_cols]
}).sort_values(["tipo_sugerido", "variable"]).reset_index(drop=True)

clasificacion_variables = {
    "numericas": summary.loc[summary["tipo_sugerido"] == "numerica", "variable"].tolist(),
    "categoricas": summary.loc[summary["tipo_sugerido"] == "categorica", "variable"].tolist(),
    "binarias": summary.loc[summary["tipo_sugerido"] == "binaria", "variable"].tolist(),
    "otras": summary.loc[summary["tipo_sugerido"] == "otro", "variable"].tolist(),
    "ordinales": []  # Completar manualmente si identificas alguna variable con orden natural
}

print("Resumen por tipo de variable:")
for tipo, cols in clasificacion_variables.items():
    print(f"- {tipo}: {len(cols)}")

display(summary)
clasificacion_variables

Resumen por tipo de variable:
- numericas: 9
- categoricas: 0
- binarias: 0
- otras: 0
- ordinales: 0


,variable,dtype,n_unicos,faltantes,tipo_sugerido
0,Left Biceps femoris caput longus,float64,3002137,0,numerica
1,Left Gastrocnemius lateralis,float64,3002137,0,numerica
2,Left Gastrocnemius medialis,float64,3002137,0,numerica
3,Left Gluteus maximus,float64,3002137,0,numerica
4,Left Semitendinosus,float64,3002137,0,numerica
5,Right Rectus femoris,float64,3002137,0,numerica
6,Right Tibialis anterior,float64,3002137,0,numerica
7,Right Vastus medialis,float64,3002137,0,numerica
8,Time,float64,1741599,0,numerica


{'numericas': ['Left Biceps femoris caput longus',
  'Left Gastrocnemius lateralis',
  'Left Gastrocnemius medialis',
  'Left Gluteus maximus',
  'Left Semitendinosus',
  'Right Rectus femoris',
  'Right Tibialis anterior',
  'Right Vastus medialis',
  'Time'],
 'categoricas': [],
 'binarias': [],
 'otras': [],
 'ordinales': []}

# Punto 2. Extraccion de caracteristicas (Feature Engineering)

En esta seccion se desarrolla:
- 2.a Ventanas de 1 segundo segun frecuencia de muestreo
- 2.b Extraccion de caracteristicas de tiempo y frecuencia
- 2.c Justificacion de las caracteristicas seleccionadas

## Punto 2.a. Ventanas de 1 segundo

Se calcula la frecuencia de muestreo a partir de la columna `Time` y luego se construyen ventanas no solapadas de 1 segundo para los 8 canales EMG.

In [17]:
# Punto 2.a: definicion de ventanas de 1 segundo
from numpy.fft import rfft, rfftfreq

# Definir columnas de senal (8 canales EMG), excluyendo tiempo y target
signal_cols = [c for c in df.columns if c not in ["Time", target_col]]
print(f"Canales detectados ({len(signal_cols)}): {signal_cols}")

# Estimar frecuencia de muestreo desde diferencias de tiempo
dt = df["Time"].diff().dropna()
dt = dt[dt > 0]
fs = int(round(1.0 / dt.median()))
window_size = fs * 1  # 1 segundo
step = window_size    # sin solapamiento

n_samples = len(df)
window_starts = np.arange(0, n_samples - window_size + 1, step)
n_windows = len(window_starts)

print(f"Frecuencia de muestreo estimada: {fs} Hz")
print(f"Tamano de ventana: {window_size} muestras")
print(f"Numero de ventanas de 1 segundo: {n_windows}")

Canales detectados (8): ['Right Rectus femoris', 'Left Gluteus maximus', 'Left Gastrocnemius medialis', 'Left Semitendinosus', 'Left Biceps femoris caput longus', 'Right Vastus medialis', 'Right Tibialis anterior', 'Left Gastrocnemius lateralis']
Frecuencia de muestreo estimada: 1000 Hz
Tamano de ventana: 1000 muestras
Numero de ventanas de 1 segundo: 3002


## Punto 2.b. Extraccion de caracteristicas por ventana y canal

Se extraen caracteristicas de tiempo y frecuencia por cada ventana de 1 segundo en cada canal:
- Tiempo: RMS, varianza, tasa de cruces por cero, pendiente media absoluta
- Frecuencia: frecuencia media, frecuencia mediana, potencia espectral total

In [18]:
# Punto 2.b: construccion de la nueva base de datos de caracteristicas
def extract_channel_features(x: np.ndarray, fs: int) -> dict:
    x = np.asarray(x, dtype=float)

    # Dominio del tiempo
    rms = np.sqrt(np.mean(x ** 2))
    var = np.var(x)
    zcr = np.mean(np.signbit(x[1:]) != np.signbit(x[:-1]))
    mean_abs_slope = np.mean(np.abs(np.diff(x)))

    # Dominio de la frecuencia (FFT)
    x_centered = x - np.mean(x)
    fft_vals = np.abs(rfft(x_centered)) ** 2
    freqs = rfftfreq(len(x_centered), d=1 / fs)

    power_sum = np.sum(fft_vals) + 1e-12
    mean_freq = np.sum(freqs * fft_vals) / power_sum
    cumsum_power = np.cumsum(fft_vals)
    median_freq = freqs[np.searchsorted(cumsum_power, 0.5 * cumsum_power[-1])]
    spectral_power = power_sum / len(fft_vals)

    return {
        "rms": rms,
        "var": var,
        "zcr": zcr,
        "mean_abs_slope": mean_abs_slope,
        "mean_freq": mean_freq,
        "median_freq": median_freq,
        "spectral_power": spectral_power,
    }

feature_rows = []
for start in window_starts:
    end = start + window_size
    w = df.iloc[start:end]

    row = {}
    for ch in signal_cols:
        feats = extract_channel_features(w[ch].values, fs)
        for fname, fvalue in feats.items():
            row[f"{ch}__{fname}"] = fvalue

    # target de ventana por mayoria
    row[target_col] = int(w[target_col].mode().iloc[0])
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)

print(f"Base de caracteristicas creada: {features_df.shape[0]} ventanas x {features_df.shape[1]} columnas")
print("Incluye target:", target_col in features_df.columns)
features_df.head()

Base de caracteristicas creada: 3002 ventanas x 57 columnas
Incluye target: True


,Right Rectus femoris__rms,Right Rectus femoris__var,Right Rectus femoris__zcr,Right Rectus femoris__mean_abs_slope,Right Rectus femoris__mean_freq,Right Rectus femoris__median_freq,Right Rectus femoris__spectral_power,Left Gluteus maximus__rms,Left Gluteus maximus__var,Left Gluteus maximus__zcr,...,Right Tibialis anterior__median_freq,Right Tibialis anterior__spectral_power,Left Gastrocnemius lateralis__rms,Left Gastrocnemius lateralis__var,Left Gastrocnemius lateralis__zcr,Left Gastrocnemius lateralis__mean_abs_slope,Left Gastrocnemius lateralis__mean_freq,Left Gastrocnemius lateralis__median_freq,Left Gastrocnemius lateralis__spectral_power,Target
0,0.011706,0.000137,0.123123,0.003076,55.440140,46.0,0.136750,0.003989,0.000016,0.122122,...,96.0,0.534211,0.025655,0.000658,0.120120,0.006508,41.731650,15.0,0.656819,0
1,0.014023,0.000197,0.111111,0.003891,57.774179,54.0,0.196263,0.004223,0.000018,0.120120,...,96.0,0.652742,0.031409,0.000986,0.149149,0.011270,78.909994,76.0,0.984443,0
2,0.014820,0.000220,0.121121,0.003840,56.110605,50.0,0.219187,0.004209,0.000018,0.126126,...,92.0,0.588205,0.026209,0.000687,0.101101,0.007284,55.350350,50.0,0.685526,0
3,0.013817,0.000191,0.121121,0.003916,56.535754,49.0,0.190535,0.004374,0.000019,0.115115,...,90.0,0.680712,0.021673,0.000470,0.154154,0.007987,80.227840,90.0,0.468743,0
4,0.013326,0.000177,0.122122,0.004042,58.254569,47.0,0.177088,0.004641,0.000022,0.120120,...,84.0,0.528279,0.025220,0.000636,0.138138,0.008710,70.649069,75.0,0.634780,0


## Punto 2.c. Justificacion de caracteristicas seleccionadas

Las caracteristicas se eligieron para capturar cambios en amplitud, variabilidad, dinamica temporal y distribucion espectral de las senales EMG relacionadas con fatiga muscular.

In [19]:
# Punto 2.c: tabla de justificacion de caracteristicas
justificacion_features = pd.DataFrame([
    {
        "caracteristica": "rms",
        "dominio": "tiempo",
        "justificacion": "Representa energia efectiva de la senal EMG; puede variar con la activacion muscular y la fatiga."
    },
    {
        "caracteristica": "var",
        "dominio": "tiempo",
        "justificacion": "Cuantifica la dispersion de amplitud; cambios en estabilidad muscular alteran su variabilidad."
    },
    {
        "caracteristica": "zcr",
        "dominio": "tiempo",
        "justificacion": "Aproxima contenido de alta frecuencia y cambios de signo en la senal."
    },
    {
        "caracteristica": "mean_abs_slope",
        "dominio": "tiempo",
        "justificacion": "Mide rapidez promedio de cambio entre muestras consecutivas."
    },
    {
        "caracteristica": "mean_freq",
        "dominio": "frecuencia",
        "justificacion": "Centro de masa espectral; suele desplazarse ante cambios fisiologicos por fatiga."
    },
    {
        "caracteristica": "median_freq",
        "dominio": "frecuencia",
        "justificacion": "Frecuencia que divide la potencia espectral en dos mitades; indicador clasico en fatiga EMG."
    },
    {
        "caracteristica": "spectral_power",
        "dominio": "frecuencia",
        "justificacion": "Sintetiza la potencia global de la ventana en el dominio espectral."
    },
])

display(justificacion_features)

print("Columnas de la nueva base (sin tiempo):", features_df.shape[1] - 1)
print("Variable objetivo conservada:", target_col)

,caracteristica,dominio,justificacion
0,rms,tiempo,Representa energia efectiva de la senal EMG; p...
1,var,tiempo,Cuantifica la dispersion de amplitud; cambios ...
2,zcr,tiempo,Aproxima contenido de alta frecuencia y cambio...
3,mean_abs_slope,tiempo,Mide rapidez promedio de cambio entre muestras...
4,mean_freq,frecuencia,Centro de masa espectral; suele desplazarse an...
5,median_freq,frecuencia,Frecuencia que divide la potencia espectral en...
6,spectral_power,frecuencia,Sintetiza la potencia global de la ventana en ...


Columnas de la nueva base (sin tiempo): 56
Variable objetivo conservada: Target
